# Example Dense Matrix Surface

Canonical dense matrix notebook for direct solve, cached matvec/rmatvec reuse, and dense operator-plan usage.

## Scope

This notebook is the canonical example surface for `example_dense_matrix_surface`. It runs against the repo source tree through `/src`, shows direct public API usage, summarizes validation and benchmark status, and includes visual summaries.

In [ ]:
import io
import json
import os
import re
import subprocess
import sys
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'arbplusjax').exists():
            return p
    raise RuntimeError(f'Could not locate repo root from: {start}')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
os.chdir(REPO_ROOT)

PYTHON = os.getenv('ARBPLUSJAX_PYTHON', sys.executable)
JAX_MODE = os.getenv('JAX_MODE', 'cpu').strip().lower()
JAX_DTYPE = os.getenv('JAX_DTYPE', 'float64').strip().lower()
RUN_ENV = os.environ.copy()
RUN_ENV['PYTHONPATH'] = str(REPO_ROOT / 'src') + os.pathsep + RUN_ENV.get('PYTHONPATH', '')
if JAX_MODE == 'cpu':
    RUN_ENV['JAX_PLATFORMS'] = 'cpu'
elif JAX_MODE == 'gpu':
    RUN_ENV['JAX_PLATFORMS'] = 'cuda'
RUN_ENV['JAX_ENABLE_X64'] = '1' if JAX_DTYPE == 'float64' else '0'
EXAMPLE_INPUT_ROOT = REPO_ROOT / 'examples' / 'inputs' / 'example_dense_matrix_surface'
EXAMPLE_OUTPUT_ROOT = REPO_ROOT / 'examples' / 'outputs' / 'example_dense_matrix_surface'
EXAMPLE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run(cmd: list[str], *, capture: bool = False):
    print('[cmd]', ' '.join(cmd))
    return subprocess.run(cmd, cwd=REPO_ROOT, env=RUN_ENV, text=True, capture_output=capture, check=True)


## Environment

The notebook reports interpreter, selected JAX mode, and the active backend/device view. Canonical retained execution in this repo state is CPU-oriented, but the notebook calling pattern remains CPU/GPU portable and explicitly parameterized for `float32` and `float64`.

In [ ]:
SUPPORTED_JAX_MODES = ('cpu', 'gpu')
SUPPORTED_JAX_DTYPES = ('float32', 'float64')
if JAX_MODE not in SUPPORTED_JAX_MODES:
    raise ValueError(f'Unsupported JAX_MODE: {JAX_MODE}')
if JAX_DTYPE not in SUPPORTED_JAX_DTYPES:
    raise ValueError(f'Unsupported JAX_DTYPE: {JAX_DTYPE}')
print('python:', PYTHON)
print('jax_mode:', JAX_MODE)
print('jax_dtype:', JAX_DTYPE)
print('supported_jax_modes:', SUPPORTED_JAX_MODES)
print('supported_jax_dtypes:', SUPPORTED_JAX_DTYPES)
print('validation_slice:', 'cpu_current__gpu_portable_contract')
runtime = run([PYTHON, 'tools/check_jax_runtime.py'], capture=True)
print(runtime.stdout)
runtime_payload = json.loads(runtime.stdout)
(EXAMPLE_OUTPUT_ROOT / f'runtime_{JAX_MODE}.json').write_text(json.dumps(runtime_payload, indent=2) + '\n', encoding='utf-8')

## Direct Usage

Construct representative real and complex dense matrices and exercise solve, matvec, cached matvec, and cached rmatvec surfaces.

In [ ]:
import jax.numpy as jnp
from arbplusjax import acb_core, acb_mat, arb_mat, double_interval as di, jcb_mat, jrb_mat

a_mid = jnp.array([[4.0, 1.0, 0.0], [2.0, 3.0, 1.0], [0.0, 1.0, 2.0]], dtype=jnp.float64)
x_mid = jnp.array([[1.0, 0.0], [2.0, 1.0], [-1.0, 3.0]], dtype=jnp.float64)
vec_mid = jnp.array([1.0, -0.5, 0.25], dtype=jnp.float64)
a = di.interval(a_mid, a_mid)
x = di.interval(x_mid, x_mid)
vec = di.interval(vec_mid, vec_mid)
rhs = arb_mat.arb_mat_matmul(a, x)
cache = arb_mat.arb_mat_dense_matvec_plan_prepare(a)
rcache = arb_mat.arb_mat_rmatvec_cached_prepare(a)
dense_plan = jrb_mat.jrb_mat_dense_operator_plan_prepare(a)

a_c_mid = a_mid + 1j * jnp.array([[0.0, 1.0, 0.0], [-1.0, 0.0, 0.5], [0.0, -0.5, 0.0]], dtype=jnp.float64)
vec_c_mid = vec_mid + 1j * jnp.array([0.25, -0.1, 0.3], dtype=jnp.float64)
a_c = acb_core.acb_box(di.interval(jnp.real(a_c_mid), jnp.real(a_c_mid)), di.interval(jnp.imag(a_c_mid), jnp.imag(a_c_mid)))
vec_c = acb_core.acb_box(di.interval(jnp.real(vec_c_mid), jnp.real(vec_c_mid)), di.interval(jnp.imag(vec_c_mid), jnp.imag(vec_c_mid)))
cache_c = acb_mat.acb_mat_dense_matvec_plan_prepare(a_c)
rcache_c = acb_mat.acb_mat_rmatvec_cached_prepare(a_c)
dense_plan_c = jcb_mat.jcb_mat_dense_operator_plan_prepare(a_c)

dense_results = {
    'solve_basic': arb_mat.arb_mat_solve(a, rhs),
    'cached_matvec': arb_mat.arb_mat_dense_matvec_plan_apply(cache, vec),
    'cached_rmatvec': arb_mat.arb_mat_rmatvec_cached_apply(rcache, vec),
    'operator_apply': jrb_mat.jrb_mat_operator_plan_apply(dense_plan, vec),
    'complex_cached_matvec': acb_mat.acb_mat_dense_matvec_plan_apply(cache_c, vec_c),
    'complex_cached_rmatvec': acb_mat.acb_mat_rmatvec_cached_apply(rcache_c, vec_c),
    'complex_operator_apply': jcb_mat.jcb_mat_operator_plan_apply(dense_plan_c, vec_c),
}
display(dense_results)

## Production Pattern

Dense production use should prepare solve and matvec/rmatvec plans once, reuse them across repeated calls, and keep dtype and batch shape stable. Dense operator plans should be the bridge into matrix-free workflows when callers later want Krylov-style execution without rewriting the model surface.

In [ ]:
rhs_batch = jnp.stack([vec, vec], axis=0)
dense_service = {
    'solve_reuse': arb_mat.arb_mat_dense_lu_solve_plan_apply(arb_mat.arb_mat_dense_lu_solve_plan_prepare(a), rhs),
    'cached_matvec': arb_mat.arb_mat_dense_matvec_plan_apply(cache, vec),
    'cached_rmatvec': arb_mat.arb_mat_rmatvec_cached_apply(rcache, vec),
    'cached_matvec_padded': arb_mat.arb_mat_dense_matvec_plan_apply_batch_padded(cache, rhs_batch, pad_to=8),
    'operator_apply': jrb_mat.jrb_mat_operator_plan_apply(dense_plan, vec),
}
display(dense_service)

## Extending Benchmarks

To extend dense benchmarks, add a stable metric in `benchmark_dense_matrix_surface.py`; use that same metric key in the matrix workbook so dense, sparse, and matrix-free surfaces remain comparable.

## Fast JAX Point Pattern

Dense point-mode fast JAX should run through the compiled public batch surface with cached-apply style operations where available.

In [ ]:
import jax
from arbplusjax import api
dense_batch = di.interval(jnp.stack([a_mid, a_mid], axis=0), jnp.stack([a_mid, a_mid], axis=0))
rhs_batch_fast = di.interval(jnp.stack([vec_mid, vec_mid], axis=0), jnp.stack([vec_mid, vec_mid], axis=0))
dense_fast = api.bind_point_batch_jit('arb_mat_matvec', dtype='float64', pad_to=4)
dense_fast_out = dense_fast(dense_batch, rhs_batch_fast)
dense_vmap = jax.vmap(lambda aa, xx: api.eval_point('arb_mat_matvec', aa, xx, dtype='float64'))(dense_batch, rhs_batch_fast)
display({'jit_shape': dense_fast_out.shape, 'jit_matches_vmap': bool(jnp.allclose(dense_fast_out, dense_vmap))})

## AD Product Pattern

Dense AD should be demonstrated in both directions on the production-facing operator-plan surface rather than only on a raw midpoint helper. This section differentiates dense operator-plan apply over both the input vector and a matrix scale parameter, then plots the paired sensitivities.

In [ ]:
import jax
base = a_mid
vec_fixed = vec_mid
def dense_loss_vec(v):
    plan = jrb_mat.jrb_mat_dense_operator_plan_prepare(di.interval(base, base))
    out = jrb_mat.jrb_mat_operator_plan_apply(plan, di.interval(v, v))
    return jnp.sum(di.midpoint(out))
def dense_loss_scale(scale):
    scaled = di.interval(scale * base, scale * base)
    plan = jrb_mat.jrb_mat_dense_operator_plan_prepare(scaled)
    out = jrb_mat.jrb_mat_operator_plan_apply(plan, vec)
    return jnp.sum(di.midpoint(out))
vec_sweep = jnp.linspace(-0.75, 0.75, 24, dtype=jnp.float64)
scale_sweep = jnp.linspace(0.75, 1.25, 24, dtype=jnp.float64)
primal_vec = jax.vmap(lambda t: dense_loss_vec(jnp.asarray([t, vec_fixed[1], vec_fixed[2]], dtype=jnp.float64)))(vec_sweep)
grad_vec = jax.vmap(lambda t: jax.grad(dense_loss_vec)(jnp.asarray([t, vec_fixed[1], vec_fixed[2]], dtype=jnp.float64))[0])(vec_sweep)
primal_scale = jax.vmap(dense_loss_scale)(scale_sweep)
grad_scale = jax.vmap(jax.grad(dense_loss_scale))(scale_sweep)
ad_df = pd.DataFrame({'vec_entry': np.asarray(vec_sweep), 'primal_vec': np.asarray(primal_vec), 'grad_vec': np.asarray(grad_vec), 'scale': np.asarray(scale_sweep), 'primal_scale': np.asarray(primal_scale), 'grad_scale': np.asarray(grad_scale)})
display(ad_df.head())
ax = ad_df.plot(x='vec_entry', y=['primal_vec', 'grad_vec'], figsize=(8, 4), title='Dense Matrix AD Validation: Argument Direction')
plt.tight_layout()
plt.savefig(EXAMPLE_OUTPUT_ROOT / f'ad_validation_argument_{JAX_MODE}.png', dpi=160, bbox_inches='tight')
plt.show()
ax = ad_df.plot(x='scale', y=['primal_scale', 'grad_scale'], figsize=(8, 4), title='Dense Matrix AD Validation: Parameter Direction')
plt.tight_layout()
plt.savefig(EXAMPLE_OUTPUT_ROOT / f'ad_validation_parameter_{JAX_MODE}.png', dpi=160, bbox_inches='tight')
plt.show()

## Validation Summary

Run the dense matrix chassis and matrix-stack contract tests that own cached matvec/rmatvec and operator-plan adaptation.

In [ ]:
tests = run([
    PYTHON, '-m', 'pytest', '-q',
    'tests/test_arb_mat_chassis.py',
    'tests/test_acb_mat_chassis.py',
    'tests/test_dense_broad_surface.py',
    'tests/test_matrix_stack_contracts.py',
], capture=True)
print(tests.stdout)
if tests.stderr:
    print(tests.stderr)
(EXAMPLE_OUTPUT_ROOT / f'pytest_{JAX_MODE}.txt').write_text(tests.stdout + ('\n' + tests.stderr if tests.stderr else ''), encoding='utf-8')

## Benchmark Summary

Run the dense matrix benchmark in schema-backed form and compare cached/direct/operator-friendly paths.

In [ ]:
dense_report = EXAMPLE_OUTPUT_ROOT / f'dense_matrix_surface_{JAX_MODE}.json'
run([PYTHON, 'benchmarks/benchmark_dense_matrix_surface.py', '--n', '8', '--warmup', '1', '--runs', '2', '--output', str(dense_report)])
bench_payload = json.loads(dense_report.read_text())
bench_df = pd.DataFrame(bench_payload['records']).sort_values('warm_time_s')
bench_df.to_csv(EXAMPLE_OUTPUT_ROOT / f'dense_benchmark_summary_{JAX_MODE}.csv', index=False)
display(bench_df[['implementation', 'operation', 'dtype', 'warm_time_s']].head(20))

## Comparison / Contrast

Summarize direct solve, cached matvec/rmatvec, and operator-plan usage so dense callers can compare production calling styles.

In [ ]:
compare_df = bench_df[bench_df['operation'].isin(['direct_solve', 'cached_matvec', 'cached_rmatvec', 'dense_plan_prepare'])].copy()
display(compare_df[['implementation', 'operation', 'warm_time_s']])

## Plots

Plot dense matrix timing by operation to make direct, cached, and operator-plan-friendly paths easy to compare visually.

In [ ]:
pivot = bench_df.pivot(index='operation', columns='implementation', values='warm_time_s')
ax = pivot.plot(kind='bar', figsize=(11, 4), title='Dense Matrix Warm Time by Operation')
ax.set_ylabel('warm_time_s')
plt.tight_layout()
plt.savefig(EXAMPLE_OUTPUT_ROOT / f'dense_benchmark_summary_{JAX_MODE}.png', dpi=160, bbox_inches='tight')
plt.show()

## Optional Diagnostics

Use the matrix stack diagnostics benchmark when compile, recompile, and operator-plan adaptation behavior needs deeper inspection.

In [ ]:
summary_lines = [
    f'# Example Dense Matrix Surface Summary ({JAX_MODE})',
    '',
    f'- backend: `{runtime_payload["platform"]}`',
    f'- benchmark_rows: `{len(bench_df)}`',
    '',
    '## Comparison Slice',
    '',
]
for row in compare_df.to_dict(orient='records'):
    summary_lines.append(f"- `{row['implementation']}` / `{row['operation']}`: warm={row['warm_time_s']:.6g}s")
(EXAMPLE_OUTPUT_ROOT / f'summary_{JAX_MODE}.md').write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')
display('\n'.join(summary_lines[:14]))